In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from data.load_data import TwitDataset, collate_fn, get_class_imbalance
from torch.utils.data import DataLoader

ds = TwitDataset()

# Train/test split
train_ds, test_ds = torch.utils.data.random_split(ds, [0.8, 0.2])
train_dataloader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)

pos_weight = get_class_imbalance(train_ds).to(device)

In [ ]:
from models import GaborNet, GaborFilter, CNNHead

N_FILTERS = 40
SAMPLE_RATE = 16000     # must match the (resampled) audio fed to the model
KERNEL_SIZE = 401       # ~25 ms @ 16 kHz: long enough to resolve ~1 kHz carriers
STRIDE = 160

feat_extract = GaborFilter(n_filters=N_FILTERS, kernel_size=KERNEL_SIZE, sample_rate=SAMPLE_RATE, stride=STRIDE)
head = CNNHead(channels=(16, 32, 64))   # 3-block 2D CNN over the [n_filters, T] map
model = GaborNet(feat_extract, head).to(device)

In [ ]:
from train import Trainer

trainer = Trainer(model, train_dataloader, test_dataloader, pos_weight, device, lr=1e-2, lr_scheduler_kwargs={"eta_min": 0.0})
trainer.train_with_slimming(
    sparsify_epochs=18,
    finetune_epochs=15,
    pruning_ratio=0.32,
    reg=1e-5,
)

In [ ]:
trainer.save_model()